# Physics-penalty run A: penalty ON  (`physics_weight = 0.1`)

Standalone session: trains ONE model on the grain-rich dataset with
`physics_weight = 0.1`, `patience = 30`, then evaluates it with the
phase-insensitive spectral metric and saves the model + eval to Google Drive.

All three run notebooks (A/B/C) use the same dataset and the same `split_seed`,
so their test sets match and the results are directly comparable. Run them in
separate sessions, then use `03d_ablation_compare.ipynb` to compare.

Set the runtime: **Runtime -> Change runtime type -> T4 GPU**, then `Runtime -> Run all`.

### 1. Mount Drive + clone repo + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard

### 2. Confirm a GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU OK:", torch.cuda.get_device_name(0))

### 3. Get the grain-rich dataset
Reuses `pfc_dataset_grains.zip` from Drive if present (instant); else regenerates it.

In [ ]:
import os
drive_zip = '/content/drive/MyDrive/pfc_dataset_grains.zip'
if os.path.exists(drive_zip):
    print('Found dataset on Drive -- unzipping.')
    !cp "$drive_zip" .
    !unzip -o -q pfc_dataset_grains.zip
else:
    print('No dataset on Drive -- generating it now.')
    !python generate_dataset.py --sweep --config pfc_config_grains.yaml --parallel --max-workers 2
    !zip -r -q pfc_dataset_grains.zip data_pfc_grains
    !cp pfc_dataset_grains.zip /content/drive/MyDrive/
import pandas as pd
m = pd.read_csv('data_pfc_grains/manifest.csv')
print(m['status'].value_counts().to_dict(), '|', len(m), 'runs')

### 4. Configure + train + evaluate
Points config at the grain dataset, sets `physics_weight = 0.1`, trains, then
evaluates. Output streams live (unbuffered). The trained model is copied to Drive
as `fno_grains_physics.pt` as soon as training finishes.

In [ ]:
import yaml, subprocess, shutil, os

TAG = 'physics'
PW  = 0.1

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['data_dir'] = 'data_pfc_grains'
cfg['data']['split_by'] = 'config'
cfg['train']['physics_weight'] = PW
out_dir = f'runs/ablation_{TAG}'
cfg['logging']['out_dir'] = out_dir
cfg['eval']['out_dir'] = f'{out_dir}/eval'
with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
print(f'===== TRAIN [{TAG}]  physics_weight={PW}  patience={cfg["train"]["patience"]} =====', flush=True)
subprocess.run(['python', '-u', 'train_fno.py', '--config', 'config.yaml'], check=True, env=env)
ckpt = f'{out_dir}/best.pt'
shutil.copy(ckpt, f'/content/drive/MyDrive/fno_grains_{TAG}.pt')
print(f'Saved model -> Drive/fno_grains_{TAG}.pt', flush=True)
print(f'===== EVAL [{TAG}] =====', flush=True)
subprocess.run(['python', '-u', 'evaluate_fno.py', '--config', 'config.yaml', '--checkpoint', ckpt], check=True, env=env)

### 5. This run's results

In [ ]:
import pandas as pd, torch
df = pd.read_csv(f'runs/ablation_{TAG}/eval/per_trajectory.csv')
ck = torch.load(f'runs/ablation_{TAG}/best.pt', map_location='cpu', weights_only=False)
print('physics_weight    :', PW)
print('best_epoch        :', ck.get('epoch'))
print('best_val_loss     :', ck.get('val_loss'))
print('mean rollout MSE  :', round(float(df['rollout_mse'].mean()), 6))
print('mean spectral MSE :', round(float(df['spectral_mse'].mean()), 6))
print('phase_offset trajs:', int(df['phase_offset'].sum()), '/', len(df))
print('\nmean rollout MSE by seed_type:')
display(df.groupby('seed_type')['rollout_mse'].mean())

In [ ]:
from IPython.display import Image, display
import glob
for png in sorted(glob.glob(f'runs/ablation_{TAG}/eval/*.png')):
    display(Image(png))

### 6. Save the evaluation to Drive

In [ ]:
!zip -r -q fno_grains_eval_{TAG}.zip runs/ablation_{TAG}/eval
!cp fno_grains_eval_{TAG}.zip /content/drive/MyDrive/
print(f'Saved to Drive:  fno_grains_{TAG}.pt  +  fno_grains_eval_{TAG}.zip')